In [0]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load propensity features
df_features = spark.table("analytics_ml.int_features.features_propensity_3").toPandas()

df_clv = spark.table("analytics_ml.int_features.features_clv_proxy")[["customerid","clv_proxy"]].toPandas()
df = df_features.merge(df_clv, on="customerid", how="left")

feature_cols = [
    "total_orders_base",
    "avg_order_value",
    "days_since_last_purchase",
    "count_email_open",
    "count_email_click",
    "count_sms_click",
    "count_push_open",
    "clv_proxy"
]

X = df[feature_cols]
y = df["label"]

if y.nunique() < 2:
    raise ValueError("The dataset must contain at least two classes for classification.")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Propensity model AUC: {auc:.3f}")

with mlflow.start_run(run_name="propensity_model_v1"):
    mlflow.log_metric("AUC", auc)
    mlflow.sklearn.log_model(model, "model")
    mlflow.log_param("model_type", "logistic_regression")

df["propensity_score"] = model.predict_proba(X)[:,1]
df["expected_revenue"] = df["propensity_score"] * df["clv_proxy"]

# Save predictions as table
spark.createDataFrame(df[["customerid","propensity_score","expected_revenue"]]) \
     .write.mode("overwrite") \
     .saveAsTable("analytics_ml.int_features.propensity_scores")

print(" Propensity scores and expected revenue saved to table: analytics_ml.mart_predictions.propensity_scores")


